In [19]:
import os
import json
import torch
import pandas as pd
import logging
import importlib
from pathlib import Path
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image as PILImage
from IPython.display import Image

import sys
PROJECT_DIR = "/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER"
SRC_DIR = str(Path(PROJECT_DIR) / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import multiomic_transformer.utils.data_formatter as data_formatter
import multiomic_transformer.utils.experiment_handler as experiment_handler

random.seed(1337)
np.random.seed(1337)
torch.manual_seed(1337)

GROUND_TRUTH_DIR = Path(PROJECT_DIR) / "data" / "ground_truth_files"
DATA_DIR = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/PROCESSED_DATA")

In [2]:
from cycler import cycler
color_palette = {
  "blue_light": "#18A6ED",
  "orange_light": "#EEA700",
  "red_light": "#EF767A",
  "green_light": "#7EE3BA",
  "purple_light": "#C798CC",
  "grey_light": "#BCBCBF",
  "blue_dark": "#2E70B9",
  "orange_dark": "#D18A3D",
  "red_dark": "#BC3E1A",
  "green_dark": "#32936F",
  "purple_dark": "#9D5ED4",
  "grey_dark": "#434B4E",
}

plt.rcParams.update({

    # figure
    "figure.figsize": (6,4),
    "figure.dpi": 300,

    # fonts
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,

    # axes
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.grid": False,
    "grid.alpha": 0.25,

    # lines
    "lines.linewidth": 2,

    # legend
    "legend.frameon": False,

    # color cycle
    "axes.prop_cycle": cycler(color=color_palette.values()),
})

method_color_dict = {
  "Gradient Attribution": "#4195DF",
  "LINGER": "#7EE3BA",
  "SCENIC+": "#EF767A",
  "CellOracle": "#F9C60D",
  "Pando": "#EF9CFA",
  "TRIPOD": "#82EC32",
  "FigR": "#FDA7BB",
  "GRaNIE": "#F98637"
}

light_colors = [v for k,v in color_palette.items() if "light" in k]

order = ["LINGER", "SCENIC+", "CellOracle", "GRaNIE", "Pando", "TRIPOD", "FigR"]

# Testing how the data prep and caching interacts with the experiment_loader class

## Load a Saved TrainingDataFormatter Object

We can load a TDF object from the disk using the `load_tdf()` function. This recreates the TDF object from a cached `settings.json` file in the experiment's processed data directory. Other attributes such as the `genes`, `tf_names`, `tg_names`, and `peaks` are loaded from the saved files in the directory. Once the TDF object is loaded, we can verify that all of the necessary files are cached properly using the `create_or_load_data_cache()` method. 

> Note: The `create_or_load_data_cache()` method is also a nice shortcut for running all of the steps to create the cached data files

In [58]:
importlib.reload(experiment_handler)

experiment_name = "Macrophage_buffer_2_raw_muon_preprocessing"

tdf = data_formatter.load_tdf(
    settings_path = Path(PROJECT_DIR) / "data" / "processed" / experiment_name / "settings.json"
)

# Verify that the data cache files exist. If not, this method will create them.
tdf.create_or_load_data_cache(sample_name=tdf.sample_names[0], force_recalculate=False)

Loaded existing settings from /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/processed/Macrophage_buffer_2_raw_muon_preprocessing/settings.json
All required files are present.
Skipping data cache creation.


## Create an ExperimentHandler

The `ExperimentHander` class uses the files and attributes from the TDF object to manage model training, gradient attribution calculations, evaluation metric calculations, and plotting. The outputs from the `ExperimentHandler` class are saved to the `experiment_dir` attribute under numbered subdirectories that separate different experiments. This allows for multiple models to be trained per experiment without overwriting the results from previous trainings.

In [64]:
# The ExperimentHandler is a higher level class that handles the training and evaluation of the model.
# It takes in a TrainingDataFormatter object to handle file paths, data loading, and caching.
logging.info("Initializing ExperimentHandler...")
exp = experiment_handler.ExperimentHandler(
    training_data_formatter=tdf,
    experiment_dir="/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/experiments/",
    model_num=1,
    silence_warnings=False,
)

Initializing ExperimentHandler...


## Create the model training loaders

Once the `ExperimentHander` is loaded, we will load in the dataset we created in Step 020 and use it to make train/val/test dataloaders for model training and evaluation.

### Creating a Multichromosome Dataset

In [ ]:
# Creates a MultiChromosomeDataset dataset class, which handles loading data for one chromosome
# at a time and caching it in memory. The max_cached argument controls how many chromosomes worth 
# of data can be cached in memory at once. Each batch only contains TG and window data from one chromosome.
# The chromosomes are loaded sequentially, starting with the first chromosome in the chrom_list.
logging.info("Creating dataset...")
exp.create_multichrom_dataset(max_cached=100)

### Preparing the Train/Val/Test DataLoaders

In [ ]:
# Prepares the Train/Val/Test dataloaders, being careful to balance the number of 
# batches from each chromosome in each set.
logging.info("Preparing DataLoader...")
train_loader, val_loader, test_loader = exp.prepare_dataloader(
    batch_size=64,
    num_workers=8
)

### Creating Data Scalers

In [ ]:
# Creates scalers for the RNA and ATAC data based on the training split.
logging.info("Creating scalers...")
exp.create_scalers(train_loader)

## MultiomicTransformer model training

In [65]:


# Creates a new MultiomicTransformer model. Model attributes can be set to change
# the hyperparameters of the model.
logging.info("Creating model")
# exp.create_new_model()

exp.load_model()

# Runs model training and returns the trained model.
# logging.info("Training model")
# model = exp.train(
#     train_loader=train_loader, 
#     val_loader=val_loader, 
#     num_epochs=500,
#     max_batches=None,
#     save_every_n_epochs=10,
#     monitor_gpu_memory=True,
#     profile_batches=True,
#     allow_overwrite=True,
#     silence_tqdm=True,
#     )

# # Runs gradient attribution to calculate the gradients between each TF input and each TG output.
# logging.info("\nRunning Gradient Attribution")
# exp.run_gradient_attribution(
#     test_loader,
#     max_batches=None,
#     max_tgs_per_batch=None,
#     )

exp.grn = exp.load_grn()

# Loads a ground truth file with columns "Source" and "Target" for TF-TG interactions.
logging.info("Loading ground truth datasets...")
GROUND_TRUTH_DIR = Path(PROJECT_DIR) / "data" / "ground_truth_files"
gt_by_dataset_dict = {
    "ChIP-Atlas macrophage": exp.load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_macrophage.csv"),
}

# Calculates the AUROC of the predicted GRN against multiple ground truth datasets.
logging.info("\nCalculating AUROC")
auroc_df = exp.calculate_auroc_all_sample_gts(exp.grn, gt_by_dataset_dict)     
logging.info(f"Pooled Median AUROC: {auroc_df['pooled_median_auroc'].iloc[0]:.3f}")       
logging.info(f"Per-TF Median AUROC: {auroc_df['per_tf_median_auroc'].iloc[0]:.3f}")

exp.save_handler()

Creating model
Loading ground truth datasets...

Calculating AUROC
Pooled Median AUROC: 0.515
Per-TF Median AUROC: 0.525


In [ ]:
from multiomic_transformer.utils import auroc_refactored
importlib.reload(auroc_refactored)

<module 'multiomic_transformer.utils.auroc_refactored' from '/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/src/multiomic_transformer/utils/auroc_refactored.py'>

In [142]:
linger_grn_dir = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/other_method_grns/LINGER_muon")
scenic_grn_dir = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/other_method_grns/SCENIC_muon")

def print_grn_stats(grn_dir, method):
    logging.info(f"\n{method} GRN stats:")
    df_dict = {
        "method": [],
        "sample_name": [],
        "tfs": [],
        "tgs": [],
        "edges": [],
    }
    
    for file in grn_dir.glob("*.tsv"):
        method_name = file.stem
        method_name = method_name.replace(method, "")
        cell_type = method_name.split("_")[0]
        sample_name = "_".join(method_name.split("_")[1:])
        
        grn_df = pd.read_csv(file, sep="\t")
        n_tfs = grn_df["Source"].nunique()
        n_tgs = grn_df["Target"].nunique()
        n_edges = len(grn_df)
        logging.info(f"  - {cell_type} {sample_name}: {n_tfs:,} TFs, {n_tgs:,} TGs, {n_edges:,} edges")
        df_dict["method"].append(method)
        df_dict["sample_name"].append(sample_name)
        df_dict["tfs"].append(n_tfs)
        df_dict["tgs"].append(n_tgs)
        df_dict["edges"].append(n_edges)
        
    return pd.DataFrame(df_dict)

scenic_grn_size_df = print_grn_stats(scenic_grn_dir, "scenic_plus")
linger_grn_size_df = print_grn_stats(linger_grn_dir, "linger")

combined_size_df = pd.concat([scenic_grn_size_df, linger_grn_size_df], ignore_index=True)


scenic_plus GRN stats:
  -  Macrophage_buffer_2: 70 TFs, 6,311 TGs, 66,400 edges
  -  mESC_E8.5_rep2: 145 TFs, 5,271 TGs, 80,969 edges
  -  K562_sample_1: 60 TFs, 6,719 TGs, 82,938 edges
  -  mESC_E7.5_rep2: 37 TFs, 3,102 TGs, 23,777 edges
  -  mESC_E8.5_rep1: 107 TFs, 4,025 TGs, 50,843 edges
  -  mESC_E7.5_rep1: 71 TFs, 3,648 TGs, 39,330 edges
  -  iPSC_WT_D13_rep1: 139 TFs, 3,411 TGs, 49,439 edges
  -  Macrophage_buffer_4: 85 TFs, 6,914 TGs, 74,651 edges
  -  Macrophage_buffer_1: 84 TFs, 6,365 TGs, 58,169 edges
  -  Macrophage_buffer_3: 73 TFs, 7,018 TGs, 84,706 edges

linger GRN stats:
  -  Macrophage_buffer_1: 395 TFs, 13,353 TGs, 5,274,435 edges
  -  mESC_E8.5_rep1: 646 TFs, 20,060 TGs, 12,958,760 edges
  -  mESC_E7.5_rep2: 574 TFs, 17,282 TGs, 9,919,868 edges
  -  mESC_E7.5_rep1: 632 TFs, 20,459 TGs, 12,930,088 edges
  -  K562_sample_1: 460 TFs, 15,290 TGs, 7,033,400 edges
  -  mESC_E8.5_rep2: 676 TFs, 22,283 TGs, 15,063,308 edges
  -  iPSC_WT_D13_rep1: 488 TFs, 17,084 TGs, 8,33

In [141]:
sample_order = [
    "mESC_E7.5_rep1",
    "mESC_E7.5_rep2",
    "mESC_E8.5_rep1",
    "mESC_E8.5_rep2",
    "Macrophage_buffer_1",
    "Macrophage_buffer_2",
    "Macrophage_buffer_3",
    "Macrophage_buffer_4",
    "K562_sample_1",
    "iPSC_WT_D13_rep1"
]

linger_grn_size_df = linger_grn_size_df.sort_values(by="sample_name", key=lambda x: x.map({name: i for i, name in enumerate(sample_order)}))
scenic_grn_size_df = scenic_grn_size_df.sort_values(by="sample_name", key=lambda x: x.map({name: i for i, name in enumerate(sample_order)}))

combined_size_df = pd.concat([scenic_grn_size_df, linger_grn_size_df], ignore_index=True)
combined_size_df

,method,cell_type,sample_name,tfs,tgs,edges
0,scenic_plus,,mESC_E7.5_rep1,71,3648,39330
1,scenic_plus,,mESC_E7.5_rep2,37,3102,23777
2,scenic_plus,,mESC_E8.5_rep1,107,4025,50843
3,scenic_plus,,mESC_E8.5_rep2,145,5271,80969
4,scenic_plus,,Macrophage_buffer_1,84,6365,58169
5,scenic_plus,,Macrophage_buffer_2,70,6311,66400
6,scenic_plus,,Macrophage_buffer_3,73,7018,84706
7,scenic_plus,,Macrophage_buffer_4,85,6914,74651
8,scenic_plus,,K562_sample_1,60,6719,82938
9,scenic_plus,,iPSC_WT_D13_rep1,139,3411,49439


In [85]:
linger_grn_dir = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/other_method_grns/LINGER_muon")

for grn_file in linger_grn_dir.glob("*.tsv"):
    logging.info(f"Loading LINGER GRN file: {grn_file.name}")

    df = pd.read_csv(grn_file, sep="\t", index_col=0).reset_index()

    cols = set(df.columns)

    # Case 1: already long
    if {"Source", "Target", "Score"}.issubset(cols):
        
        if df["Source"].nunique() > df["Target"].nunique():
            logging.info(f"  - Renaming columns for {grn_file.name}")
            df = df.rename(columns={"Source": "Target", "Target": "Source"})
            df = df[["Source", "Target", "Score"]]
            df.to_csv(grn_file, sep="\t", index=False)
        else:
            logging.info(f"  - {grn_file.name} already appears to be in long format. Skipping.")
        
        continue

    # Case 2: likely wide
    source_col = df.columns[0]

    # Heuristic: wide GRN matrix usually has many columns beyond the first identifier column
    if len(df.columns) > 3:
        df_long = (
            df.melt(
                id_vars=source_col,
                var_name="Target",
                value_name="Score"
            )
            .rename(columns={source_col: "Source"})
        )

        # Optional: remove missing values
        df_long = df_long.dropna(subset=["Score"])

        out_file = grn_file.with_name(grn_file.stem + "_long.tsv")
        df_long.to_csv(out_file, sep="\t", index=False)
        logging.info(f"  - Saved melted GRN to {out_file.name}")
    else:
        logging.warning(f"  - Could not confidently determine format for {grn_file.name}. Skipping.")

Loading LINGER GRN file: linger_Macrophage_buffer_1.tsv
Renaming columns for linger_Macrophage_buffer_1.tsv
Loading LINGER GRN file: linger_mESC_E8.5_rep1.tsv
Renaming columns for linger_mESC_E8.5_rep1.tsv
Loading LINGER GRN file: linger_mESC_E7.5_rep2.tsv
Renaming columns for linger_mESC_E7.5_rep2.tsv
Loading LINGER GRN file: linger_mESC_E7.5_rep1.tsv
Renaming columns for linger_mESC_E7.5_rep1.tsv
Loading LINGER GRN file: linger_K562_sample_1.tsv
Renaming columns for linger_K562_sample_1.tsv
Loading LINGER GRN file: linger_mESC_E8.5_rep2.tsv
Renaming columns for linger_mESC_E8.5_rep2.tsv
Loading LINGER GRN file: linger_iPSC_WT_D13_rep1.tsv
Renaming columns for linger_iPSC_WT_D13_rep1.tsv
Loading LINGER GRN file: linger_Macrophage_buffer_3.tsv
Renaming columns for linger_Macrophage_buffer_3.tsv
Loading LINGER GRN file: linger_Macrophage_buffer_2.tsv
Renaming columns for linger_Macrophage_buffer_2.tsv
Loading LINGER GRN file: linger_Macrophage_buffer_4.tsv
Renaming columns for linger_Ma

## Loading a saved ExperimentHandler

In [16]:
importlib.reload(experiment_handler)
importlib.reload(data_formatter)

<module 'multiomic_transformer.utils.data_formatter' from '/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/src/multiomic_transformer/utils/data_formatter.py'>

In [6]:
def combine_images_with_spans(
    layout,
    space,
    fig_dir,
    output_name="combined_images.png",
    bg_color=(255, 255, 255, 255),
):
    """
    Combine images into a manually positioned grid with optional row/column spanning.

    Parameters
    ----------
    layout : dict
        Dictionary mapping image path -> config dict

        Required keys per image:
            row : int
            col : int

        Optional keys per image:
            scale : float, default 1.0
            rowspan : int, default 1
            colspan : int, default 1

        Example:
        {
            Path("a.png"): {"row": 0, "col": 0, "scale": 1.0, "rowspan": 1, "colspan": 2},
            Path("b.png"): {"row": 0, "col": 2, "scale": 0.8},
            Path("c.png"): {"row": 1, "col": 0, "scale": 1.1, "rowspan": 2, "colspan": 2},
        }

    space : int
        Pixels between grid cells.
    fig_dir : Path
        Output directory.
    output_name : str
        Output filename.
    bg_color : tuple
        RGBA background color.

    Returns
    -------
    Path
        Path to the saved combined image.
    """
    items = []

    # Load images and parse layout
    for image_path, cfg in layout.items():
        row = cfg["row"]
        col = cfg["col"]
        scale = cfg.get("scale", 1.0)
        rowspan = cfg.get("rowspan", 1)
        colspan = cfg.get("colspan", 1)

        if rowspan < 1 or colspan < 1:
            raise ValueError(f"{image_path}: rowspan and colspan must be >= 1")

        img = PILImage.open(image_path).convert("RGBA")
        new_width = max(1, int(img.width * scale))
        new_height = max(1, int(img.height * scale))
        img_resized = img.resize((new_width, new_height), PILImage.LANCZOS)

        items.append({
            "path": image_path,
            "row": row,
            "col": col,
            "rowspan": rowspan,
            "colspan": colspan,
            "img": img_resized,
            "width": new_width,
            "height": new_height,
        })

    if not items:
        raise ValueError("Layout is empty.")

    n_rows = max(item["row"] + item["rowspan"] for item in items)
    n_cols = max(item["col"] + item["colspan"] for item in items)

    # Check for overlapping occupied cells
    occupied = {}
    for item in items:
        for r in range(item["row"], item["row"] + item["rowspan"]):
            for c in range(item["col"], item["col"] + item["colspan"]):
                key = (r, c)
                if key in occupied:
                    raise ValueError(
                        f"Overlap detected: {item['path']} overlaps with {occupied[key]} at cell {key}"
                    )
                occupied[key] = item["path"]

    # Compute minimum column widths and row heights
    # Start from single-cell requirements
    col_widths = [0] * n_cols
    row_heights = [0] * n_rows

    for item in items:
        if item["colspan"] == 1:
            col_widths[item["col"]] = max(col_widths[item["col"]], item["width"])
        if item["rowspan"] == 1:
            row_heights[item["row"]] = max(row_heights[item["row"]], item["height"])

    # Expand widths/heights as needed for spanned items
    # A simple iterative pass is usually enough for plotting layouts
    for _ in range(10):
        changed = False

        for item in items:
            # Width across spanned columns, including inter-column spaces inside the span
            current_span_width = (
                sum(col_widths[item["col"]: item["col"] + item["colspan"]])
                + space * (item["colspan"] - 1)
            )
            if current_span_width < item["width"]:
                deficit = item["width"] - current_span_width
                add_per_col = deficit / item["colspan"]
                for c in range(item["col"], item["col"] + item["colspan"]):
                    new_val = int(round(col_widths[c] + add_per_col))
                    if new_val > col_widths[c]:
                        col_widths[c] = new_val
                        changed = True

            # Height across spanned rows, including inter-row spaces inside the span
            current_span_height = (
                sum(row_heights[item["row"]: item["row"] + item["rowspan"]])
                + space * (item["rowspan"] - 1)
            )
            if current_span_height < item["height"]:
                deficit = item["height"] - current_span_height
                add_per_row = deficit / item["rowspan"]
                for r in range(item["row"], item["row"] + item["rowspan"]):
                    new_val = int(round(row_heights[r] + add_per_row))
                    if new_val > row_heights[r]:
                        row_heights[r] = new_val
                        changed = True

        if not changed:
            break

    # Compute canvas size
    background_width = sum(col_widths) + space * (n_cols - 1)
    background_height = sum(row_heights) + space * (n_rows - 1)

    background = PILImage.new("RGBA", (background_width, background_height), bg_color)

    # Precompute grid start coordinates
    col_starts = []
    x = 0
    for w in col_widths:
        col_starts.append(x)
        x += w + space

    row_starts = []
    y = 0
    for h in row_heights:
        row_starts.append(y)
        y += h + space

    # Paste each image centered within its spanned rectangle
    for item in items:
        x0 = col_starts[item["col"]]
        y0 = row_starts[item["row"]]

        span_width = (
            sum(col_widths[item["col"]: item["col"] + item["colspan"]])
            + space * (item["colspan"] - 1)
        )
        span_height = (
            sum(row_heights[item["row"]: item["row"] + item["rowspan"]])
            + space * (item["rowspan"] - 1)
        )

        x_offset = (span_width - item["width"]) // 2
        y_offset = (span_height - item["height"]) // 2

        paste_x = x0 + x_offset
        paste_y = y0 + y_offset

        background.paste(item["img"], (paste_x, paste_y), item["img"])

    output_path = fig_dir / output_name
    background.save(output_path)
    return output_path

def generate_all_plots(exp: experiment_handler.ExperimentHandler, gt_by_dataset_dict_sample: dict):
    logging.info(f"\n----- GRN Overlap with Ground Truth -----")
    for ground_truth_name, ground_truth in gt_by_dataset_dict_sample.items():
        exp.report_grn_overlap_with_gt(ground_truth_name, ground_truth)

    logging.info(f"\n----- AUROC -----")
    per_tf_df_all =  pd.read_csv(exp.model_training_dir / "per_tf_auroc_auprc_results.csv")
    pooled_df_all = pd.read_csv(exp.model_training_dir / "pooled_auroc_auprc_results.csv")

    per_tf_plot_df = (
        per_tf_df_all.dropna(subset=["auroc"])
        .groupby(['method', 'gt'], as_index=False)
        .agg(
            auroc=('auroc', 'median'),
        )
    )

    pooled_df_median_auroc = pooled_df_all[pooled_df_all["method"] == "Gradient Attribution"]["auroc"].median()
    per_tf_median_auroc = per_tf_plot_df[per_tf_plot_df["method"] == "Gradient Attribution"]["auroc"].median()

    logging.info(f"Median Pooled AUROC: {pooled_df_median_auroc:.3f}")
    logging.info(f"Median Per-TF AUROC: {per_tf_median_auroc:.3f}")

    fig_dir = exp.model_training_dir / "figures"
    fig_dir.mkdir(exist_ok=True)

    ground_truth_name = list(gt_by_dataset_dict_sample.keys())[0]

    name = exp.experiment_name.replace("_", " ")
    per_tf_plot, per_tf_df, tf_curves = exp.plot_top_n_tf_roc_curves(
        exp.grn, 
        gt_by_dataset_dict_sample[ground_truth_name], 
        ground_truth_name, 
        exp, 
        method_name="Gradient Attribution", 
        num_top_tfs_to_plot=10,
        min_edges=500,
        min_pos=50,
        balance=True,
        name_clean=name,
        override_title=f"{ground_truth_name}\nTop 10 Per-TF AUROC"
        )
    per_tf_plot.savefig(fig_dir / f"top_10_per_tf_auroc.png", dpi=250, bbox_inches="tight")
    plt.close(per_tf_plot)

    pooled_auroc_boxplot_fig = exp._plot_all_results_auroc_boxplot(
        pooled_df_all,
        per_tf=False,
        ylim=(0.2, 0.8),
        override_title=f"Pooled AUROC per Method",
        method_color_dict=exp.method_color_dict
    )
    pooled_auroc_boxplot_fig.savefig(fig_dir / f"pooled_auroc_boxplot.png", dpi=250, bbox_inches="tight")
    plt.close(pooled_auroc_boxplot_fig)

    per_tf_auroc_boxplot_fig = exp._plot_all_results_auroc_boxplot(
        per_tf_plot_df, 
        per_tf=True,
        ylim=(0.2, 0.8),
        override_title=f"Per-TF AUROC per Method",
        method_color_dict=exp.method_color_dict
        )
    per_tf_auroc_boxplot_fig.savefig(fig_dir / f"per_tf_auroc_boxplot.png", dpi=250, bbox_inches="tight")
    plt.close(per_tf_auroc_boxplot_fig)

    relative_improvement_fig = exp.plot_relative_improvement(
        per_tf_plot_df, 
        exp.experiment_name,
        override_title=f"Per-TF AUROC Improvement",
        )
    relative_improvement_fig.savefig(fig_dir / f"relative_improvement.png", dpi=250, bbox_inches="tight")
    plt.close(relative_improvement_fig)

    pooled_auroc_heatmap_fig = exp.plot_method_gt_heatmap(
        pooled_df_all, 
        per_tf=False,
        x_scale=1.2,
        y_scale=0.6,
        override_title=f"Pooled AUROC by Method and GT"
        )
    pooled_auroc_heatmap_fig.savefig(fig_dir / f"pooled_auroc_heatmap.png", dpi=250, bbox_inches="tight")
    plt.close(pooled_auroc_heatmap_fig)

    per_tf_auroc_heatmap_fig = exp.plot_method_gt_heatmap(
        per_tf_plot_df, 
        per_tf=True,
        x_scale=1.2,
        y_scale=0.6,
        override_title=f"Per-TF AUROC by Method and GT"
        )
    per_tf_auroc_heatmap_fig.savefig(fig_dir / f"per_tf_auroc_heatmap.png", dpi=250, bbox_inches="tight")
    plt.close(per_tf_auroc_heatmap_fig)

    true_vs_predicted_fig = exp.plot_true_vs_predicted_tg_expression(
        num_batches=len(exp.test_loader), 
        set_axis_logscale=False,
        title=f"Predicted vs True TG Expression"
        )
    true_vs_predicted_fig.savefig(fig_dir / f"true_vs_predicted.png", dpi=250, bbox_inches="tight")
    plt.close(true_vs_predicted_fig)
    
    layout = {
        fig_dir / "top_10_per_tf_auroc.png": {
            "row": 0, "col": 0, "scale": 0.9, "rowspan": 1, "colspan": 2
        },
        fig_dir / "pooled_auroc_boxplot.png": {
            "row": 0, "col": 2, "scale": 0.9
        },
        fig_dir / "pooled_auroc_heatmap.png": {
            "row": 0, "col": 3, "scale": 1.0
        },
        fig_dir / "true_vs_predicted.png": {
            "row": 1, "col": 0, "scale": 0.85
        },
        fig_dir / "relative_improvement.png": {
            "row": 1, "col": 1, "scale": 0.85
        },
        fig_dir / "per_tf_auroc_boxplot.png": {
            "row": 1, "col": 2, "scale": 0.9
        },
        fig_dir / "per_tf_auroc_heatmap.png": {
            "row": 1, "col": 3, "scale": 1.0
        },
    }

    output_path = combine_images_with_spans(
        layout=layout,
        space=50,
        fig_dir=fig_dir,
        output_name="combined_images.png",
    )
    
    return output_path

def rank_methods_by_auroc(df):
    """Calculates the sorted rank by median AUROC for each inference method"""
    df_grouped = (
        df
        .groupby("method", as_index=False)
        .agg(median_auroc=("auroc", "median"))
        .sort_values(by="median_auroc", ascending=False)
        .reset_index(drop=True)
        .reset_index(drop=False)
        .rename(columns={"index": "rank"})
        .assign(rank=lambda x: x["rank"] + 1)
    )
    return df_grouped[["method", "median_auroc", "rank"]]

def get_method_rank(df, method_name):
    """Calculates the sorted rank by median AUROC for an inference method"""
    df_grouped = rank_methods_by_auroc(df)
    method_rank = df_grouped[df_grouped["method"] == method_name]["rank"].values[0]
    return method_rank 

def avg_rank_by_method_plot(avg_rank_df, method_color_dict, title, rename_map=None):
    fig = plt.figure(figsize=(6, 4))

    order = avg_rank_df["method"].tolist()

    sns.barplot(
        data=avg_rank_df,
        x="method",
        y="avg_rank",
        order=order,
        palette=method_color_dict
    )

    ax = plt.gca()
    
    if rename_map is None:
        rename_map = {}

    # Rename + recolor ticks
    new_labels = []
    for tick in ax.get_xticklabels():
        original = tick.get_text()
        new = rename_map.get(original, original)
        new_labels.append(new)

        # Only color MTGRN
        if new == "MTGRN":
            tick.set_color(method_color_dict.get(original, "black"))
            tick.set_fontweight("bold")
        else:
            tick.set_color("black")
            tick.set_fontweight("normal")

    ax.set_xticklabels(new_labels, rotation=45, ha="right")

    plt.ylabel("Average Rank")
    plt.xlabel("")
    plt.title(title)
    plt.tight_layout()
    return fig

def experiment_by_method_rank_heatmap(all_ranks_df, rank_df, method_color_dict, experiment_order, rename_map=None):
    if rename_map is None:
        rename_map = {}

    rank_heatmap_df = all_ranks_df.pivot(
        index="experiment",
        columns="method",
        values="rank"
    )

    method_order = rank_df["method"].tolist()

    rank_heatmap_df = rank_heatmap_df.reindex(
        index=experiment_order,
        columns=method_order
    )

    fig, ax = plt.subplots(figsize=(8, 6))

    sns.heatmap(
        rank_heatmap_df,
        annot=True,
        fmt=".0f",
        cmap="viridis_r",
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "Rank"},
        ax=ax
    )

    # Rename + recolor ticks
    new_labels = []
    for tick in ax.get_xticklabels():
        original = tick.get_text()
        new = rename_map.get(original, original)
        new_labels.append(new)

        if new == "MTGRN":
            tick.set_color(method_color_dict.get(original, "black"))
            tick.set_fontweight("bold")
        else:
            tick.set_color("black")
            tick.set_fontweight("normal")

    ax.set_xticklabels(new_labels, rotation=45, ha="right")

    ax.set_title("Method Rank by Experiment")
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()

    return fig

def per_tf_method_rank_plots(selected_experiments, method_color_dict, rename_map=None):

    all_ranks = []

    for sample_name, experiment_name, sample_type, model_num in selected_experiments:

        exp = experiment_handler.load_experiment_handler(
            tdf_settings_path=DATA_DIR / experiment_name / "settings.json",
            experiment_dir=Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/EXPERIMENTS/"),
            model_num=model_num,
        )
        if not (exp.model_training_dir / "per_tf_auroc_auprc_results.csv").exists():
            logging.warning(f"{exp.experiment_name} does not have per-TF results. Skipping.")
            continue
        
        per_tf_df_all =  pd.read_csv(exp.model_training_dir / "per_tf_auroc_auprc_results.csv")

        per_tf_df_all = per_tf_df_all[per_tf_df_all["method"] != "TF Knockout"]

        per_tf_plot_df = (
            per_tf_df_all.dropna(subset=["auroc"])
            .groupby(['method', 'gt'], as_index=False)
            .agg(
                auroc=('auroc', 'median'),
            )
        )
        
        per_tf_df_grouped = (
            per_tf_plot_df
            .groupby("method", as_index=False)
            .agg(median_auroc=("auroc", "median"))
            .sort_values(by="median_auroc", ascending=False)
            .reset_index(drop=True)
            .reset_index(drop=False)
            .rename(columns={"index": "rank"})
            .assign(rank=lambda x: x["rank"] + 1)
        )

        # Add experiment identifier
        per_tf_df_grouped["experiment"] = sample_name

        all_ranks.append(per_tf_df_grouped[["experiment", "method", "rank"]])
        
    all_ranks_df = pd.concat(all_ranks, ignore_index=True)

    final_ranking = (
        all_ranks_df
        .groupby("method", as_index=False)
        .agg(
            total_rank=("rank", "sum"),
            avg_rank=("rank", "mean"),
            n_experiments=("rank", "count")
        )
        .sort_values(by="total_rank")  # lower = better
        .reset_index(drop=True)
    )

    # display(final_ranking)

    fig = avg_rank_by_method_plot(final_ranking, method_color_dict, title="Average Method Ranking by Per-TF AUROC", rename_map=rename_map)
    fig.show()

    experiment_order = [sample_name for sample_name, _, _, _ in selected_experiments]
    fig = experiment_by_method_rank_heatmap(all_ranks_df, final_ranking, method_color_dict, experiment_order=experiment_order, rename_map=rename_map)
    fig.show()
    
def pooled_method_rank_plots(selected_experiments, method_color_dict, rename_map=None):
    all_ranks = []

    for sample_name, experiment_name, sample_type, model_num in selected_experiments:

        exp = experiment_handler.load_experiment_handler(
            tdf_settings_path=DATA_DIR / experiment_name / "settings.json",
            experiment_dir=Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/EXPERIMENTS/"),
            model_num=model_num,
        )
        if not (exp.model_training_dir / "pooled_auroc_auprc_results.csv").exists():
            logging.warning(f"{exp.experiment_name} does not have pooled results. Skipping.")
            continue

        pooled_df_all = pd.read_csv(exp.model_training_dir / "pooled_auroc_auprc_results.csv")

        pooled_df_all = pooled_df_all[pooled_df_all["method"] != "TF Knockout"]

        pooled_df_grouped = rank_methods_by_auroc(pooled_df_all)

        # Add experiment identifier
        pooled_df_grouped["experiment"] = sample_name

        all_ranks.append(pooled_df_grouped[["experiment", "method", "rank"]])
        
    all_ranks_df = pd.concat(all_ranks, ignore_index=True)

    final_ranking = (
        all_ranks_df
        .groupby("method", as_index=False)
        .agg(
            total_rank=("rank", "sum"),
            avg_rank=("rank", "mean"),
            n_experiments=("rank", "count")
        )
        .sort_values(by="total_rank")  # lower = better
        .reset_index(drop=True)
    )

    # display(final_ranking)

    fig = avg_rank_by_method_plot(final_ranking, method_color_dict, title="Average Method Ranking by Pooled AUROC", rename_map=rename_map)
    fig.show()

    experiment_order = [sample_name for sample_name, _, _, _ in selected_experiments]
    fig = experiment_by_method_rank_heatmap(all_ranks_df, final_ranking, method_color_dict, experiment_order=experiment_order, rename_map=rename_map)
    fig.show()

In [ ]:
def load_ground_truth(ground_truth_file: str|Path):
    if type(ground_truth_file) == str:
        ground_truth_file = Path(ground_truth_file)
        
    if ground_truth_file.suffix == ".csv":
        sep = ","
    elif ground_truth_file.suffix == ".tsv":
        sep="\t"
        
    ground_truth_df = pd.read_csv(ground_truth_file, sep=sep, on_bad_lines="skip", engine="python")
    
    if "chip" in ground_truth_file.name and "atlas" in ground_truth_file.name:
        ground_truth_df = ground_truth_df[["source_id", "target_id"]]

    if ground_truth_df.columns[0] != "Source" or ground_truth_df.columns[1] != "Target":
        ground_truth_df = ground_truth_df.rename(columns={ground_truth_df.columns[0]: "Source", ground_truth_df.columns[1]: "Target"})
    ground_truth_df["Source"] = ground_truth_df["Source"].astype(str).str.upper()
    ground_truth_df["Target"] = ground_truth_df["Target"].astype(str).str.upper()
    
    # Build TF, TG, and edge sets for quick lookup later
    gt = ground_truth_df[["Source", "Target"]].dropna()

    gt_tfs = set(gt["Source"].unique())
    gt_tgs = set(gt["Target"].unique())
    
    gt_pairs = (gt["Source"] + "\t" + gt["Target"]).drop_duplicates()
    
    gt_lookup = (gt_tfs, gt_tgs, set(gt_pairs))
        
    return ground_truth_df, gt_lookup

gt_by_dataset_dict = {
    "Macrophage": {
        # "RN204": load_ground_truth(GROUND_TRUTH_DIR / "rn204_macrophage_human_chipseq.tsv"),
        "ChIP-Atlas macrophage": load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_macrophage.csv"),
    },
    "mESC": {
        "ChIP-Atlas mESC": load_ground_truth(GROUND_TRUTH_DIR / "chip_atlas_tf_peak_tg_dist.csv"),
        "RN111": load_ground_truth(GROUND_TRUTH_DIR / "RN111.tsv"),
        "RN112": load_ground_truth(GROUND_TRUTH_DIR / "RN112.tsv"),
        "RN114": load_ground_truth(GROUND_TRUTH_DIR / "RN114.tsv"),
        "RN116": load_ground_truth(GROUND_TRUTH_DIR / "RN116.tsv"),        
    },
    "K562": {
        "ChIP-Atlas K562": load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_K562.csv"),
        "RN117": load_ground_truth(GROUND_TRUTH_DIR / "RN117.tsv"),        
    },
    "iPSC": {
        # "ChIP-Atlas iPSC": load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_iPSC.csv"),
        "ChIP-Atlas iPSC (1 Mb)": load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_iPSC_1mb.csv"),
        # "ChIP-Atlas iPSC (100 kb)": load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_iPSC_100kb.csv"),
    }
}

In [ ]:
experiment_name = "mESC_E7.5_rep2_full_pipeline"
sample_type = "mESC"


exp = experiment_handler.load_experiment_handler(
    tdf_settings_path=DATA_DIR / experiment_name / "settings.json",
    experiment_dir=Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/EXPERIMENTS/"),
    model_num=2,
    )

gt_by_dataset_dict_sample = gt_by_dataset_dict[sample_type]

# output_path = generate_all_plots(exp, gt_by_dataset_dict_sample)
# Image(str(output_path))

Loaded existing settings from /gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/PROCESSED_DATA/mESC_E7.5_rep2_full_pipeline/settings.json
Loading ExperimentHandler state from /gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/EXPERIMENTS/mESC_E7.5_rep2_full_pipeline/model_training_002/experiment_handler_save.json...


In [ ]:
exp.calculate_auroc_all_methods(
    exp.tdf.sample_names, 
    "mESC", 
    gt_by_dataset_dict_sample, 
    grn=exp.grn,
    use_muon_grn=True
)

### Plots for ranking all methods

In [ ]:
selected_preprocessing_experiments = [
    ("mESC E7.5_rep1", "mESC_E7.5_rep1_full_pipeline", "mESC", 2),
    ("mESC E7.5_rep2", "mESC_E7.5_rep2_full_pipeline", "mESC", 2),
    ("mESC E8.5_rep1", "mESC_E8.5_rep1_full_pipeline", "mESC", 2),
    ("mESC E8.5_rep2", "mESC_E8.5_rep2_full_pipeline", "mESC", 2),
    ("Macrophage_buffer_1", "Macrophage_buffer_1_full_pipeline", "Macrophage", 2),
    ("Macrophage_buffer_2", "Macrophage_buffer_2_full_pipeline", "Macrophage", 2),
    ("K562", "K562_sample_1_full_pipeline", "K562", 2),
    ("iPSC", "iPSC_WT_D13_rep1_full_pipeline", "iPSC", 2)
    ]

# rename_map = {"Gradient Attribution": "MTGRN"}
rename_map = None
    
pooled_method_rank_plots(selected_preprocessing_experiments, method_color_dict, rename_map=rename_map)
per_tf_method_rank_plots(selected_preprocessing_experiments, method_color_dict, rename_map=rename_map)

In [21]:
importlib.reload(experiment_handler)

<module 'multiomic_transformer.utils.experiment_handler' from '/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/src/multiomic_transformer/utils/experiment_handler.py'>

In [ ]:
import multiomic_transformer.utils.auroc_refactored as auroc_utils
selected_preprocessing_experiments = [
    ("E7.5_rep1", "mESC_E7.5_rep1_full_pipeline", "mESC", 2),
    ("E7.5_rep2", "mESC_E7.5_rep2_full_pipeline", "mESC", 2),
    # ("E8.5_rep1", "mESC_E8.5_rep1_full_pipeline", "mESC", 2),
    # ("E8.5_rep2", "mESC_E8.5_rep2_full_pipeline", "mESC", 2),
    # ("buffer_1", "Macrophage_buffer_1_full_pipeline", "Macrophage", 2),
    # ("buffer_2", "Macrophage_buffer_2_full_pipeline", "Macrophage", 2),
    # ("sample_1", "K562_sample_1_full_pipeline", "K562", 2),
    # ("WT_D13_rep1", "iPSC_WT_D13_rep1_full_pipeline", "iPSC", 2)
    ]

def evaluate_single_method_setwise(
        method_name: str,
        score_df: pd.DataFrame,
        ground_truth: tuple[pd.DataFrame, dict[str, set[str]]],
        ground_truth_name: str,
        max_edges: int = 10000,
    ) -> dict | None:
        """
        Mimics the R evaluate_single_method() logic:

        1. Uppercase TF/target names
        2. Filter inferred network to GT TFs and GT targets
        3. Take top max_edges edges overall
        4. Compare inferred edge set to GT edge set
        5. Compute precision, recall, F1

        Returns
        -------
        dict with:
            - metrics: dict(precision, recall, f1)
            - summary: one-row DataFrame
            - final_edges: filtered top-edge DataFrame
        """
        ground_truth_df, gt_lookup = ground_truth

        if score_df is None or len(score_df) == 0:
            return None

        # ---- Build GT TF/target sets ----
        gt_tfs = set(ground_truth_df["Source"].astype(str).str.upper())
        gt_targets = set(ground_truth_df["Target"].astype(str).str.upper())

        # GT edge pairs as strings, matching the R behavior
        gt_pairs = set(
            ground_truth_df["Source"].astype(str).str.upper()
            + "_"
            + ground_truth_df["Target"].astype(str).str.upper()
        )

        # ---- Filter inferred network by GT TFs and GT targets ----
        df_filtered = score_df.copy()
        df_filtered["source_upper"] = df_filtered["Source"].astype(str).str.upper()
        df_filtered["target_upper"] = df_filtered["Target"].astype(str).str.upper()

        df_filtered = df_filtered[
            df_filtered["source_upper"].isin(gt_tfs)
            & df_filtered["target_upper"].isin(gt_targets)
        ].copy()

        # ---- Sort by score descending before taking top max_edges ----
        if "Score" in df_filtered.columns:
            df_filtered = df_filtered.sort_values("Score", ascending=False, kind="stable")

        n_edges = min(max_edges, len(df_filtered))
        if n_edges == 0:
            print(f"    {method_name}: No edges after filtering, skipping")
            return None

        df_final = df_filtered.head(n_edges).copy()

        # ---- Create inferred edge pairs ----
        inferred_pairs = set(
            df_final["source_upper"].astype(str) + "_" + df_final["target_upper"].astype(str)
        )

        # ---- Set-based evaluation ----
        tp = inferred_pairs.intersection(gt_pairs)
        fp = inferred_pairs.difference(gt_pairs)
        fn = gt_pairs.difference(inferred_pairs)

        tp_n = len(tp)
        fp_n = len(fp)
        fn_n = len(fn)

        precision = tp_n / (tp_n + fp_n) if (tp_n + fp_n) > 0 else 0.0
        recall = tp_n / (tp_n + fn_n) if (tp_n + fn_n) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

        inferred_tfs = set(df_final["source_upper"].unique())
        inferred_targets = set(df_final["target_upper"].unique())

        summary_df = pd.DataFrame([{
            "Method": method_name,
            "GT_name": ground_truth_name,
            "GT_total_edges": len(gt_pairs),
            "GT_TFs": len(gt_tfs),
            "GT_targets": len(gt_targets),
            "Original_network_size": len(score_df),
            "Filtered_network_size": len(df_filtered),
            "Final_network_size": len(df_final),
            "Inferred_TFs": len(inferred_tfs),
            "Inferred_targets": len(inferred_targets),
            "TP": tp_n,
            "FP": fp_n,
            "FN": fn_n,
            "Precision": round(precision, 4),
            "Recall": round(recall, 4),
            "F1_score": round(f1, 4),
            "Common_TFs": len(gt_tfs.intersection(inferred_tfs)),
            "Common_targets": len(gt_targets.intersection(inferred_targets)),
        }])

        return summary_df

method_dfs = []
other_method_dict = {}
logging.info(f"Loading other method GRNs for all samples")
for sample_name, _, sample_type, _ in selected_preprocessing_experiments:
    sample_other_method_dict = auroc_utils.load_other_method_muon_grns([sample_name], sample_type)
    other_method_dict[sample_name] = sample_other_method_dict

logging.info(f"\nCalculating evaluation metrics for each sample")
for sample_name, experiment_name, sample_type, model_num in selected_preprocessing_experiments:
    experiment_dir = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/EXPERIMENTS/") / experiment_name / f"model_training_{model_num:03d}"
    
    grn_df = pd.read_csv(experiment_dir / "inferred_grn.csv", index_col=None)
    standardized_method_dict = other_method_dict.get(sample_name, {}).copy()
    
    standardized_method_dict["Gradient Attribution"] = grn_df

    for gt_name, ground_truth in gt_by_dataset_dict_sample.items():
        logging.info(f"  - Evaluating methods for {sample_name} against GT: {gt_name}")
        for method_name, grn_df in standardized_method_dict.items():
            result = evaluate_single_method_setwise(
                method_name=method_name,
                score_df=grn_df,
                ground_truth=ground_truth,
                ground_truth_name=gt_name,
                max_edges=10000,
            )
            
            result["sample_name"] = sample_name
            
            if result is not None:
                method_dfs.append(result)
        
combined_df = pd.concat(method_dfs, ignore_index=True)
combined_df.head()


Processing sample: E7.5_rep1 | Dataset: mESC
  - Loading SCENIC+
  - Loading LINGER

Processing sample: E7.5_rep2 | Dataset: mESC
  - Loading SCENIC+
  - Loading LINGER
Evaluating methods for E7.5_rep1 against GT: ChIP-Atlas mESC
Evaluating methods for E7.5_rep1 against GT: RN111
Evaluating methods for E7.5_rep1 against GT: RN112
Evaluating methods for E7.5_rep1 against GT: RN114
Evaluating methods for E7.5_rep1 against GT: RN116
Evaluating methods for E7.5_rep2 against GT: ChIP-Atlas mESC
